In [0]:
dbutils.widgets.dropdown(name='environment',defaultValue='dev',choices=['dev','prod','qa'],label='select Environment')
env=dbutils.widgets.get('environment')
print(env)


In [0]:
 silvTable=f"salesLake_{env}.silver_{env}.CleanseMonthlysale"
 print(silvTable)
 bronzetable=f'saleslake_{env}.bronze_{env}.rawmonthlysales'
 print(bronzetable)


In [0]:
spark.sql(f'''
insert into {silvTable}
select distinct  
  cast(trim(sale_id) as integer) as sale_id, 
  upper(product) as product,
  upper(category) as category,
  cast(trim(quantity) as integer) as quantity,
  cast(trim(price) as double) as price,
  try_to_date(trim(sales_date), 'yyyy-MM-dd') as sales_date, 
  upper(region) as region,
  current_timestamp() as ingest_ts
from {bronzetable}
where ingest_ts >
  (select coalesce(max(ingest_ts), to_timestamp('1900-01-01', 'yyyy-MM-dd')) from {silvTable})
order by cast(trim(sale_id) as integer)
''')